# Neural Networks - Coding

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

# Power Trading Data

## Load data

Data is provided by [Helios Power Trading](https://heliospowertrading.com/)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/My Drive/Colab Notebooks' + '/de_data.xlsx' # You will have to change the path such that it corresponds to the location where you have th file saved

In [ ]:
df = pd.read_excel(path)

In [ ]:
df

SpotDE: Spot price of power in Germany, Euro pr mega watt hour

ConDE: Consumption (forecast) of power in Germany

WndDE: Wind production (forecast)

SolarDE: Solar production (forecast)

RdlDE: Residual load (Consumption -(solar + wind))

TempDE: Temperature

## Data exploration

**Basic Statistics**

In [ ]:
print(df['SpotDE'].describe(), '\n')
print(df['TempDE'].describe())

**Line plots**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['DeliveryTime'], df['SpotDE'], color='b', linestyle='-', linewidth=2)

# Adding labels and title
plt.xlabel('Sample Index')
plt.ylabel('Price (SpotDE)')
plt.title('Price (SpotDE) Over Samples')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['DeliveryTime'], df['TempDE'], color='b', linestyle='-', linewidth=1)

# Adding labels and title
plt.xlabel('Sample Index')
plt.ylabel('Temperature (TempDE)')
plt.title('Temperature (TempDE) Over Samples')
plt.grid(True)
plt.show()

**Histograms**

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['SpotDE'], bins=100, color='g', edgecolor='k', alpha=0.7)

# Adding labels and title
plt.xlabel('Price (SpotDE)')
plt.ylabel('Frequency')
plt.title('Distribution of Price (SpotDE)')
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['TempDE'], bins=100, color='g', edgecolor='k', alpha=0.7)

# Adding labels and title
plt.xlabel('Temperature (TempDE)')
plt.ylabel('Frequency')
plt.title('Distribution of Temperature (TempDE)')
plt.grid(axis='y', linestyle='--')
plt.show()

## Cleaning Data

**Removal of observations with missing values, outliers etc.**

First a function for finding NA values are create

In [ ]:
def extractfunc(x):
  return [index for index, ele in enumerate(x) if math.isnan(ele)]

Using Pandas .apply method, we apply it to all the columns, except the first. From that we get a list of the indices with a missing value

In [ ]:
tmpoutput = df.iloc[:,1:].apply(extractfunc, 0)

We bind all these indices to a list with no duplicates. We create a list of the opposite indices

In [ ]:
combinedlist = []
for i in tmpoutput:
  combinedlist = list(set(combinedlist + i))
toget = [i for i in range(len(df)) if i not in combinedlist]

We select all observations with no missing values in any of the variables. Data is cleaned

In [ ]:
dfcleaned = df.iloc[toget,:].reset_index(drop=True)


*   SpotDE: Spot price in euro per mega watt hour
*   ConDE: Consumption
*   WndDE: Wind production
*   SolarDE: Sun production
*   RdlDE: Residual load (consumption - (sun + wind))
*   TempDE: Temperature

In [ ]:
dfcleaned

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(dfcleaned['DeliveryTime'], dfcleaned['SpotDE'], color='b', linestyle='-', linewidth=2)

# Adding labels and title
plt.xlabel('Sample Index')
plt.ylabel('Price (SpotDE)')
plt.title('Price (SpotDE) Over Samples')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(dfcleaned['DeliveryTime'], dfcleaned['TempDE'], color='b', linestyle='-', linewidth=1)

# Adding labels and title
plt.xlabel('Sample Index')
plt.ylabel('Temperature (TempDE)')
plt.title('Temperature (TempDE) Over Samples')
plt.grid(True)
plt.show()

## Transformations and Scaling

Function for transforming the DeliveryTime variable

In [ ]:
def transformtime(x):
  year = x.year
  month = x.month
  day = x.day
  hour = x.hour
  day_of_week = x.weekday()
  is_weekend = 1 if day_of_week >= 5 else 0
  return [year, month, day, hour, day_of_week, is_weekend]

Apply the transformation

In [ ]:
transformlist = []
for time in dfcleaned.loc[:,'DeliveryTime']:
  transformlist.append(transformtime(time))

Then concatenate this with the original data frame

In [ ]:
columns = ['Year', 'Month', 'Day', 'Hour', 'Day_of_Week', 'Is_Weekend']
dftime = pd.DataFrame(transformlist, columns=columns)

In [ ]:
dfcleaned = pd.concat([dfcleaned, dftime], axis = 1).drop(columns = ['DeliveryTime'])

Then the transform the Day_of_Week variable to dummies. Transform the Hour and Month variables using sine and cosine.

$$
x = \sin\left(2 \cdot \pi \frac{x}{24}\right)
$$

$$
x = \cos\left(2 \cdot \pi \frac{x}{24}\right)
$$

In [ ]:
dfcleaned = pd.get_dummies(dfcleaned, columns=['Day_of_Week'], drop_first=False)
dfcleaned['Hour_sin'] = np.sin(2 * np.pi * dfcleaned['Hour'] / 24)
dfcleaned['Hour_cos'] = np.cos(2 * np.pi * dfcleaned['Hour'] / 24)
dfcleaned['Month_sin'] = np.sin(2 * np.pi * dfcleaned['Month'] / 12)
dfcleaned['Month_cos'] = np.cos(2 * np.pi * dfcleaned['Month'] / 12)

In [ ]:
dfcleaned = dfcleaned.drop(columns=['Month', 'Hour'])

In [ ]:
dfcleaned

## Training & Testing

**Training**

First split the data into explanatory variables and target variable

In [ ]:
X = dfcleaned.drop(columns = ['SpotDE'])
y = dfcleaned.loc[:,'SpotDE']

In [ ]:
np.floor(len(X) * 0.8)

In [ ]:
X_train = X.iloc[0:9722,]
X_test =  X.iloc[9722:,]
y_train = y.iloc[0:9722,]
y_test = y.iloc[9722:,]

Scale the data. Though some of the variables are kept out of scaling. The mean and standard deviation of the training set are also used to scale the test set.

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[['ConDE', 'WndDE', 'SolarDE', 'RdlDE', 'TempDE']] = scaler.fit_transform(X_train[['ConDE', 'WndDE', 'SolarDE', 'RdlDE', 'TempDE']])
X_test_scaled[['ConDE', 'WndDE', 'SolarDE', 'RdlDE', 'TempDE']] = scaler.transform(X_test[['ConDE', 'WndDE', 'SolarDE', 'RdlDE', 'TempDE']])

In [ ]:
X_train_scaled

**Build the model**

In [ ]:
from sklearn.neural_network import MLPRegressor #https://scikit-learn.org/dev/modules/generated/sklearn.neural_network.MLPRegressor.html
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

The size of the neural network is defined using 'hidden_layer_sizes=()'.

*   A hidden_layer_sizes=(10,) means that there is one hidden layer with 10 neurons
*   A hidden_layer_sizes=(10,50,) means that there are two hidden layers, the first has 10 neurons, and the second 50.

In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=(10,50))

Below we build a network of 3 hidden layers, 150 neurson in the first, 100 in the second and 50 in the third.

Some of the tuning parameters:
*   Number of iterations: Set using the 'max_iter' variable.
*   Batch size: Set using the 'batch_size' variable.
*   Random_state variable: Set such that the results can be replicated. Default is 'None', so if not set you might get different results every time you train the network.
*   Verbose: Report the current iteration and loss.
*   learning_rate_init (Learning rate): Default is 0.001.
*   alpha (L2 regularization) : Default is 0.0001



In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=(150,100,50,), max_iter=25, batch_size = 20, random_state=42, verbose=True, learning_rate_init = 0.001)

**Train the model**

Use the .fit() function to train the model

In [ ]:
mlp.fit(X_train_scaled, y_train)

**Test the model**

In [ ]:
y_pred = mlp.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error (MSE): {mse:.2f}")

In [ ]:
plt.plot(y_test.values, label='Actual Values', color='b', linestyle='-', alpha=0.6)

# Plot predicted values
plt.plot(y_pred, label='Predicted Values', color='r', linestyle='--', alpha=0.8)

plt.xlabel('Sample Index')
plt.ylabel('SpotDE (Power Price)')
plt.title('Actual vs Predicted Values')
plt.legend()
plt.show()

# California Housing dataset

Two points about this data set.

*   Look at another example of how to transform/scale data
*   See how one can under-/overfit




## Data exploration with transformation/scaling

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target

In [ ]:
df

We plot histograms of all the variables to see their distribution.

*   Latitude: Latitude of the block group's centroid (geographical coordinate).
*   Longitude: Longitude of the block group's centroid (geographical coordinate).

In [ ]:
df.hist(bins=30, figsize=(15, 10))
plt.suptitle("Histograms of California Housing Dataset Features", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

We see that MedInc is positive (right) skewed. So we take log to it to make it more symmetric.

In [ ]:
df['MedInc'] = np.log1p(df['MedInc'])

plt.figure(figsize=(6, 4))
plt.hist(df['MedInc'], bins=30, color='skyblue', edgecolor='black')
plt.title("Histogram of Log-Transformed Median Income (MedInc)")
plt.xlabel("Log(Median Income)")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Next we scale all the features using standardization, except Latitude and Logitude. We scale these using normalization as these are coordinates.

In [ ]:
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

# standardize some of the features
standardize_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
df[standardize_features] = scaler_standard.fit_transform(df[standardize_features])

# Apply Min-Max scaling Latitude and Longitude
minmax_features = ['Latitude', 'Longitude']
df[minmax_features] = scaler_minmax.fit_transform(df[minmax_features])

In [ ]:
df.hist(bins=30, figsize=(15, 10))
plt.suptitle("Histograms of California Housing Dataset Features", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Training and Testing

Above was just exploration of the data. The data above would not be suitable for training/testing as scaling had been applied for the whole data set, and there could be a potential data leakage. Instead we need to first split the data, and then apply the transformations we saw above.

In [ ]:
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['MedHouseVal'] = data.target

# Split the data into training and testing sets
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Apply log transformation to the MedInc variable in both the training and test set.
X_train['MedInc'] = np.log1p(X_train['MedInc'])
X_test['MedInc'] = np.log1p(X_test['MedInc'])

scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

# Define which features to standardize and normalize
standardize_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
minmax_features = ['Latitude', 'Longitude']

# Fit the standard scaler on the training data and transform both training and test sets
X_train[standardize_features] = scaler_standard.fit_transform(X_train[standardize_features])
X_test[standardize_features] = scaler_standard.transform(X_test[standardize_features])

# Fit the Min-Max scaler on the training data and transform both training and test sets
X_train[minmax_features] = scaler_minmax.fit_transform(X_train[minmax_features])
X_test[minmax_features] = scaler_minmax.transform(X_test[minmax_features])

We don't apply standardization or normalization to the target variable. However, we can apply log transformation, if the target variable is positive skewed or has long right tails. Doing so makes it more symmetric, closer to normal disitribution, and makes it easier for the model to learn the relationship between the features and the target.

In [ ]:
# Log-transform the target variable if needed
y_train = np.log1p(y_train)  # Only if you need to log-transform the target
y_test = np.log1p(y_test)    # Transform test target similarly for evaluation purposes

In [ ]:
hidden_layer_configs = [
    (10,),              # Very simple model (underfitting likely)
    (50,),              # Medium complexity model
    (100,),             # Higher complexity
    (50, 50),           # Two hidden layers
    (100, 50, 25),      # Deeper network
    (100, 100, 50, 25)  # Very deep network (overfitting likely)
]

train_errors = []
test_errors = []

for config in hidden_layer_configs:
    mlp = MLPRegressor(hidden_layer_sizes=config, max_iter=500, random_state=42)
    mlp.fit(X_train, y_train)

    train_mse = mean_squared_error(y_train, mlp.predict(X_train))
    test_mse = mean_squared_error(y_test, mlp.predict(X_test))

    train_errors.append(train_mse)
    test_errors.append(test_mse)

In [ ]:
for index, config in enumerate(hidden_layer_configs):
  print(f"Configuration {config} | Train MSE: {train_errors[index]:.4f}, Test MSE: {test_errors[index]:.4f}")

In [ ]:
# Plot the training and test errors for each model configuration
plt.figure(figsize=(10, 6))
plt.plot([str(config) for config in hidden_layer_configs], train_errors, label="Training Error", marker='o')
plt.plot([str(config) for config in hidden_layer_configs], test_errors, label="Test Error", marker='o')
plt.xlabel("Hidden Layer Configuration")
plt.ylabel("Mean Squared Error")
plt.title("Effect of Model Complexity on Training and Test Errors")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# A Simple Multiclass Classification Example

We wanna build a network that can predict written digits

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

In [ ]:
X[0]

In [ ]:
image = X[0].reshape(28, 28)

In [ ]:
image

In [ ]:
# Plot the image
plt.imshow(image, cmap='gray')
plt.title("Actual Label: {}".format(y[0]))
plt.axis('off')
plt.show()

Split the data and scale it

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

And train it using the 'MLPClassifier' function

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(50,), max_iter=20, random_state=42, verbose=True)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

In [ ]:
indices = range(0, 5)

plt.figure(figsize=(12, 3))
for i, index in enumerate(indices):
    image = X_test[index].reshape(28, 28)
    plt.subplot(1, len(indices), i + 1)
    plt.imshow(image, cmap='gray')
    plt.title(f"Label: {y_test[index]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# Task: Work with either Telco Customer Churn data, Bank Marketing Dataset or Financial Data

The idea with this task is for you to try to go through the different steps of 'buildling' a neural network.

*   Explore the data - What type of data is it? What type of variables are avaiable? What type of transformation/scaling is needed?
*   Clean the data - Is there some inconsistency, outliers, some variables that need formatting? etc.
*   Need for any feature engineering? Create new variables from already exsisting variables. This could be particullarly relevant for the financial data.
*   Split the data into training and test, and apply the necessary transformations/scaling. You can also set up cross validation if you want, remember that here the k-1 folds are scaled and the last fold, i.e. validation set, here is scaled with the same values as the k-1 folds.
*   Build the model, train it and test it.

**Telco Customer Churn**

Build a model that can predict whether a customer churned with the dataset 'WA_Fn-UseC_-Telco-Customer-Churn.xls'

**Bank Marketing**

Similar as Teclo Customer Churn, predict whether a customer sucribed with the dataset 'bank_marketing.csv'

**Financial Data**

Predict the sentiment (up or down next day) or price/return of a stock or index

You can incorporate more variable by calculating different statistics such as moving average etc. predicting prices/returns of stocks and indices is hard, so don't expect really good results.

Some code to get started with. The data is available through the yfinance package, so you need to install it first

In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf

# Download historical data for S&P 500 ETF (SPY)
# You can also choose other ETF's or stocks such as APPL (Apple), MSFT (Microsoft), NVDA (Nvidia).
# Data on a lot of different stocks is availabe. To my knowledge at least all of the stocks in the 500 index.
ticker = 'SPY'  # S&P 500 ETF as a proxy for the S&P 500 index
data = yf.download(ticker, start="2015-01-01", end="2023-01-01")

# Create target labels for 'Up' (1) and 'Down' (0) based on daily closing price changes
data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)

# Drop the last row, as it will have a NaN target (no next-day data to compare)
data = data.dropna()

# Define features (you can choose more technical indicators here for a more complex model)
features = ['Open', 'High', 'Low', 'Close', 'Volume']

# Create the feature set X and the target variable y
X = data[features]
y = data['Target']

In [ ]:
X